# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and can be used to programmatically discover, download, and analyze the data.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

## Load Croissant dataset (schema + data interface)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print(f"Version: {getattr(metadata, 'version', 'N/A')}")


## 2. Data Overview
Review the available record sets and fields discovered in this Croissant dataset. The dataset may contain one or more record sets (tables/views).

We'll enumerate all `recordSet` entities, printing their `@id`, `name`, and `description`, and then list their fields (columns) and associated `@id`s.

In [ ]:
# List RecordSets and fields by their @id
record_sets = dataset.record_sets  # List of mlc.RecordSet
print(f"Discovered {len(record_sets)} record sets.\n")
overview = {}

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {getattr(rs, 'description', '')}")
    print("  Fields (columns):")
    fields = rs.fields
    field_ids = []
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        field_ids.append(field.id)
    overview[rs.id] = field_ids
    print()

print("Summary of all RecordSets and field @id's:")
print(json.dumps(overview, indent=2))

## 3. Data Extraction
Load data from a specific record set as a DataFrame for analysis, referencing all sets and fields by their `@id`.

Choose a record set of interest. For this demonstration, we'll use the main protein quantification table, which typically contains fields for protein accession, name, abundant peptides, MW (molecular weight), and normalized abundance.

**Update the `target_record_set_id` below based on the printed RecordSet `@id` above as appropriate for your exploration.**

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Example: Select the primary protein record set by @id
# IF YOU SEE NO DATAFRAME BELOW, ADJUST target_record_set_id to an actual table
target_record_set_id = record_set_ids[0] if record_set_ids else None

if target_record_set_id is not None:
    for rsid in record_set_ids:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} rows for RecordSet {rsid}.")
    print(f"\nColumns in record set '{target_record_set_id}':")
    print(dataframes[target_record_set_id].columns.tolist())
    display(dataframes[target_record_set_id].head())
else:
    print("No record sets were found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Explore numeric variables, filter records, normalize a selection, and group by a categorical field.
All fields are referenced by their `@id` as printed previously.

In [ ]:
# --- Example: Numeric processing for this dataset ---

# CHANGE BELOW IF NEEDED: Pick an actual numeric field @id
numeric_field = None
group_field = None
if target_record_set_id is not None and not dataframes[target_record_set_id].empty:
    # Try to auto-select a numeric field: find the first float- or int-like column
    df = dataframes[target_record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # Try to auto-select a group field
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and len(df[col].unique()) < min(20, len(df)//5):
            group_field = col
            break
    if numeric_field is not None:
        threshold = df[numeric_field].quantile(0.75)  # Use 75%ile as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by '{group_field}' (mean {numeric_field}):")
            display(grouped_df.to_frame().head())
        else:
            print('No suitable group field for grouping.')
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Fields are referenced by their `@id`.

For example, visualize the distribution of the main numeric variable and group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id is not None and not dataframes[target_record_set_id].empty and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[target_record_set_id][numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(
            x=group_field, y=numeric_field, data=dataframes[target_record_set_id],
        )
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No plottable data found.")

## 6. Conclusion

In this notebook, we've:
- Loaded a Croissant dataset via `mlcroissant`
- Explored the record sets and their field structure using `@id` references
- Loaded the data as DataFrames and performed example processing
- Visualized and summarized numeric distributions

You can now modify field selections, filtering logic, and visualizations to match your specific research or analysis objectives with the FAIR^2 dataset.